# LangChain + Ollama

이 노트북은 Mac의 `mlx` Conda 환경에서 JupyterLab으로 실행하는 예제입니다. 로컬 Ollama에 설치한 `qwen3.6:27b` 모델을 LangChain으로 호출해 질문과 응답을 하는 기능을 구현해 봅니다.

# Ollama 모델 연결

`ChatOllama`는 기본적으로 `http://localhost:11434`의 Ollama 서버에 연결합니다. `temperature=0`은 같은 요청에 대해 비교적 일관된 형식을 얻기 위한 설정이고, `num_predict`는 최대 생성 토큰 수입니다.

In [1]:
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

In [2]:
MODEL_NAME = "qwen3.5:9b-mlx"

chat_llm = ChatOllama(
    model=MODEL_NAME,
    base_url="http://localhost:11434",
    temperature=0,
)

# Langchain Q&A

In [3]:
output_parser = StrOutputParser()

In [4]:
cot_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "사용자의 질문에 단계적으로 답변하세요."),
        ("human", "{question}"),
    ]
)

cot_chain = cot_prompt | chat_llm | output_parser

In [5]:
output = cot_chain.invoke({"question": "10 + 2 * 3"})
print(output)

주어진 식 **10 + 2 * 3** 을 연산 순서 (우선순위) 에 따라 단계별로 계산해 보겠습니다.

1. **곱셈 우선 계산**: 덧셈과 곱셈이 모두 있을 때, 먼저 곱셈을 수행합니다.
   $$2 \times 3 = 6$$

2. **덧셈 수행**: 첫 번째 단계의 결과 (6) 를 10 에 더합니다.
   $$10 + 6 = 16$$

따라서 최종 답은 **16** 입니다.


In [6]:
summarize_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "단계적으로 생각한 답변에서 결론만 추출하세요."),
        ("human", "{text}"),
    ]
)

summarize_chain = summarize_prompt | chat_llm | output_parser

In [7]:
cot_summarize_chain = cot_chain | summarize_chain
output = cot_summarize_chain.invoke({"question": "10 + 2 * 3"})
print(output)

16


# 함수를 Chain에 붙이기

In [8]:
from langchain_core.runnables import RunnableLambda, chain

In [9]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        ("human", "{input}"),
    ]
)

In [10]:
def upper(text: str) -> str:
    return text.upper()

In [11]:
chain_lambda = prompt | chat_llm | output_parser | RunnableLambda(upper)

In [ ]:
ai_message = chain_lambda.invoke({"input": "Hello!"})
print(ai_message)

In [ ]:
@chain
def upper_deco(text: str) -> str:
    return text.upper()

In [ ]:
chain_deco = prompt | chat_llm | output_parser | upper_deco

In [ ]:
ai_message = chain_deco.invoke({"input": "Hello!"})
print(ai_message)